Per day: roll travel (15% chance) → determines the day's single country (home or travel)

Logins that day: weekday 1-3, weekend 0-1 — all using that day's chosen country

Timezone for the 9-5±1hr window: always the entity's home country's timezone, regardless of which country they're logged in from

Per-login fields: source_ip (Faker), source_device, login_result (mostly success), mfa_used (mostly true), vpn_detected (mostly false) — exact probability splits not yet decided, reasonable defaults fine for v1

In [0]:
pip install Faker, pycountry

In [0]:
import uuid
import datetime
import random
import pycountry
import pytz
import zoneinfo
from faker import Faker
from pyspark.sql.functions import col

event_id	UUID (PK)	

system_identifier	string	email

login_timestamp	timestamp	

source_ip	string	

source_country	string	Direct field — real auth systems vary on whether they enrich at capture; treated as 
already-enriched here (see decisions log)

vpn_detected	boolean	

mfa_used	boolean	

login_result	string	success / failure

source_device	string	e.g. "Windows/Chrome", "iOS/Safari"

In [0]:
# List of 44 european countries
europe_countries = [
    "Albania", "Andorra", "Austria", "Belarus", "Belgium", 
    "Bosnia and Herzegovina", "Bulgaria", "Croatia", "Cyprus", "Czechia", 
    "Denmark", "Estonia", "Finland", "France", "Germany", 
    "Greece", "Hungary", "Iceland", "Ireland", "Italy", 
    "Latvia", "Liechtenstein", "Lithuania", "Luxembourg", "Malta", 
    "Moldova", "Monaco", "Montenegro", "Netherlands", "North Macedonia", 
    "Norway", "Poland", "Portugal", "Romania", "Russia", 
    "San Marino", "Serbia", "Slovakia", "Slovenia", "Spain", 
    "Sweden", "Switzerland", "Ukraine", "United Kingdom"
]

# Devices specs
devices_list = ["Windows/Chrome", "Mac/Chrome", "Android/Chrome", "iOS/Safari", "Windows/Edge", "Windows/Brave"]
devices_weights = [0.3, 0.2, 0.1, 0.1, 0.1, 0.2]

# Login specs 
login_list = ["Success", "Failure"]
login_weights = [0.95, 0.05]

# MFA specs
mfa_list = [True, False]
mfa_weights = [0.9, 0.1]

# VPN specs
vpn_list = [True, False]
vpn_weights = [0.03, 0.97]

# Travel specs
travel_list = [True, False]
travel_weights = [0.15, 0.85]

fake = Faker()

In [0]:
human_entity_df = spark.read.table("entity_risk_platform.seed_data.entities").filter(col("entity_type") == "human").select("entity_id", "home_country")

auth_identifier_df = spark.read.table("entity_risk_platform.seed_data.entity_system_identifiers").filter(col("system_name") == "auth")

auth_entity_df = auth_identifier_df.join(human_entity_df, "entity_id")

# auth_entity_df.show()

auth_entities = [row.asDict() for row in auth_entity_df.collect()]

In [0]:
europe_timzones = {}

for ec in europe_countries:
    country = pycountry.countries.search_fuzzy(ec)[0]
    country_code = country.alpha_2
    country_timezone = pytz.country_timezones.get(country_code)[0]
    europe_timzones[ec] = country_timezone

# print(europe_timzones)

In [0]:
# chosen_entity = random.choice(auth_entities)
# print(chosen_entity)

In [0]:
def generate_login_country(home_country):
    if(random.choices(travel_list, weights=travel_weights)[0]):
        return random.choice([c for c in europe_countries if c != home_country]) 
    else:
        return home_country

In [0]:
def generate_login_timestamp(date, country):
    # 8 AM
    min_seconds = 8 * 3600 
    # 6 PM
    max_seconds = 18 * 3600
    login_date = datetime.datetime.combine(date.date(), datetime.time.min) + datetime.timedelta(seconds=random.randrange(min_seconds, max_seconds))
    local_date = login_date.replace(tzinfo=zoneinfo.ZoneInfo(europe_timzones[country]))
    utc_date = local_date.astimezone(zoneinfo.ZoneInfo("UTC"))
    return utc_date

# print(generate_login_timestamp(datetime.datetime.now(), chosen_entity["home_country"]))

In [0]:
def generate_login_event_count(date):
    weekno = date.weekday()
    if weekno < 5:
        event_count = random.randrange(1, 4)
    else:
        event_count = random.randrange(0, 2)
    return event_count

# print(generate_login_event_count(datetime.datetime.now()))

In [0]:
target_date = datetime.datetime.now()

In [0]:
login_event_list = []

In [0]:
def generate_login_events(date, entities):
    events = []
    for chosen_entity in entities:
        login_country = generate_login_country(chosen_entity["home_country"])
        login_events = generate_login_event_count(date)
        for _ in range(login_events):
            events.append({
                "event_id": str(uuid.uuid4()),
                "entity_id": chosen_entity["entity_id"],
                "system_identifier": chosen_entity["system_identifier"],
                "login_timestamp": generate_login_timestamp(date, chosen_entity["home_country"]),
                "source_ip": fake.ipv4(),
                "source_device": random.choices(devices_list, weights=devices_weights)[0],
                "login_result": random.choices(login_list, weights=login_weights)[0],
                "mfa_used": random.choices(mfa_list, weights=mfa_weights)[0],
                "vpn_detected": random.choices(vpn_list, weights=vpn_weights)[0],
                "source_country": login_country,
            })
    return events

login_event_list = generate_login_events(target_date, auth_entities)
print(login_event_list)